In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr


In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 64
num_quantiles = 5
samples_per_quantile = 60
top_k_hardest = 3
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
    "num_quantiles": num_quantiles,
    "samples_per_quantile": samples_per_quantile,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

quantile_labels = [f"Q{i+1}" for i in range(num_quantiles)]
df["label_quantile"] = pd.qcut(df["label"], q=num_quantiles, labels=quantile_labels, duplicates="drop")
df["label_quantile"] = df["label_quantile"].astype(str)
df = df.sort_values(["label_quantile", "label", "sentence1", "sentence2"]).reset_index(drop=True)

slice_df = (
    df.groupby("label_quantile", group_keys=False, sort=True)
      .head(samples_per_quantile)
      .reset_index(drop=True)
      .copy()
)

slice_df["group_index"] = slice_df.groupby("label_quantile").cumcount()

quantile_summary = (
    df.groupby("label_quantile", dropna=False)
      .agg(full_count=("label", "size"), label_min=("label", "min"), label_max=("label", "max"), label_mean=("label", "mean"))
      .reset_index()
)

slice_summary = (
    slice_df.groupby("label_quantile", dropna=False)
        .agg(slice_count=("label", "size"), slice_label_min=("label", "min"), slice_label_max=("label", "max"), slice_label_mean=("label", "mean"))
        .reset_index()
)

print({"full_num_examples": len(df), "slice_num_examples": len(slice_df), "columns": slice_df.columns.tolist()})
print(quantile_summary.to_string(index=False))
print(slice_summary.to_string(index=False))
print(slice_df.head())


In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)


In [ ]:
sentences1 = slice_df["sentence1"].tolist()
sentences2 = slice_df["sentence2"].tolist()
labels = slice_df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)


In [ ]:
def safe_corr(y_true, y_pred, fn):
    if len(y_true) < 2:
        return np.nan
    if np.allclose(np.std(y_true), 0) or np.allclose(np.std(y_pred), 0):
        return np.nan
    return float(fn(y_pred, y_true).statistic if fn is spearmanr else fn(y_pred, y_true)[0])

results_df = slice_df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])

spearman_corr = safe_corr(labels, predicted_score_0_5, spearmanr)
pearson_corr = safe_corr(labels, predicted_score_0_5, pearsonr)
mae = float(np.mean(np.abs(predicted_score_0_5 - labels)))
rmse = float(np.sqrt(np.mean((predicted_score_0_5 - labels) ** 2)))
score_mean = float(np.mean(predicted_score_0_5))
score_std = float(np.std(predicted_score_0_5))
label_mean = float(np.mean(labels))
label_std = float(np.std(labels))

print(results_df[["label_quantile", "sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "absolute_error"]].head(10).to_string(index=False))


In [ ]:
per_quantile_metrics = []

for quantile_name, group in results_df.groupby("label_quantile", sort=True):
    y_true = group["label"].to_numpy(dtype=np.float32)
    y_pred = group["predicted_score_0_5"].to_numpy(dtype=np.float32)
    per_quantile_metrics.append({
        "label_quantile": quantile_name,
        "count": int(len(group)),
        "label_min": float(group["label"].min()),
        "label_max": float(group["label"].max()),
        "label_mean": float(group["label"].mean()),
        "pred_mean": float(group["predicted_score_0_5"].mean()),
        "pred_std": float(group["predicted_score_0_5"].std(ddof=0)),
        "mae": float(np.mean(np.abs(y_pred - y_true))),
        "rmse": float(np.sqrt(np.mean((y_pred - y_true) ** 2))),
        "spearman": safe_corr(y_true, y_pred, spearmanr),
        "pearson": safe_corr(y_true, y_pred, pearsonr),
    })

per_quantile_metrics_df = pd.DataFrame(per_quantile_metrics)
print(per_quantile_metrics_df.to_string(index=False))


In [ ]:
for quantile_name, group in results_df.groupby("label_quantile", sort=True):
    hardest = group.nlargest(top_k_hardest, "absolute_error")[
        ["sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "absolute_error"]
    ].reset_index(drop=True)
    print(f"Hardest examples for {quantile_name}:")
    print(hardest.to_string(index=False))
    print()


In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"full_num_examples: {len(df)}")
print(f"slice_num_examples: {len(results_df)}")
print(f"num_quantiles: {results_df['label_quantile'].nunique()}")
print(f"overall_spearman_correlation: {spearman_corr:.6f}")
print(f"overall_pearson_correlation: {pearson_corr:.6f}")
print(f"overall_mae: {mae:.6f}")
print(f"overall_rmse: {rmse:.6f}")
print(f"predicted_score_mean: {score_mean:.6f}")
print(f"predicted_score_std: {score_std:.6f}")
print(f"label_mean: {label_mean:.6f}")
print(f"label_std: {label_std:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
